# Epic 4 — `claim_rules.db` build notebook

Builds the database that powers Epic 4 (Claim Verification): given a product and a
nutrition/health claim printed on its packaging, decide whether that claim is
supported, not supported, or cannot currently be verified with the data on hand,
citing the specific FSANZ provision applied.

## Source documents (all in `claim_data/`, all read in full this session)

| File | What it is | Registration | Effective |
|---|---|---|---|
| `Schedule4_Nutrition_Health_Related_Claims_2026-04-30.pdf` | Numeric threshold tables (S4-3 nutrition content claims, S4-4/S4-5 health claims, S4-6 NPSC pass/fail lines) | F2026C00415 | 30 Apr 2026 |
| `F2025C00833.pdf` | Standard 1.2.7 — the operative provisions Schedule 4's tables are appendices to (what counts as a claim, general restrictions, procedural conditions) | F2025C00833 | 13 Aug 2025 |
| `F2022C01053.pdf` | Schedule 5 — the actual NPSC calculation method (baseline/V/P/F points → final score formula) | F2022C01053 | 1 Sep 2022 |
| `F2024C00903.pdf` | Standard 2.9.2 — infant-specific overrides for a subset of claims | F2024C00903 | 13 Sep 2024 |
| `getting-your-claims-right.pdf` | Non-legal practical guide (informs UI/decision-tree design, not a legal source) | — | Feb 2018 (amendments only to No. 174 — treat as reference for flow only, not for current numbers) |

## How the five documents relate

Standard 1.2.7 is the operative law — it defines what a claim is, sets blanket
restrictions (e.g. no claims on infant formula, kava, or high-alcohol food), and says
*when* a numeric condition applies, but contains no numbers itself. Every numeric
table it points to ("For subsection 1.2.7-12(1), the table is:") lives in Schedule 4.
Schedule 4's own NPSC pass/fail line (S4-6, e.g. "Category 2 <4") is itself scored
using a calculation method that lives in a third document, Schedule 5 — baseline
points (energy/sat fat/sugar/sodium) minus V points (fruit/veg/nut/legume content)
minus P points (protein) minus F points (fibre). Standard 2.9.2 sits alongside all of
this as a population-specific override: for products that meet its legal definition
of "food for infants" (narrower than Food Trust's overall infant/young-children
product scope — it excludes infant formula, meal replacements, formulated
supplementary foods, and unprocessed fruit/veg), a handful of properties (protein,
vitamins/minerals) get a different numeric threshold than Schedule 4's general rule,
and two absolute prohibited-representation rules apply that have no Schedule 4
equivalent at all.

## Design: 3 tables, not the originally-proposed 3+ separately-typed set

An earlier proposal split claim rules across three tables by *legal category*
(numeric rules / prohibited products / prohibited representations). The user
correctly pushed back — two of those tables would have held 1-2 rows each. This build
instead splits by **data shape**, which only produces a genuine 3-way split:

- **`claim_rules`** — every claim-eligibility rule, both numeric-threshold rows and
  non-numeric absolute-prohibition rows, discriminated by a `rule_type` column
  (`numeric_threshold`, `not_permitted`, `prohibited_representation`,
  `requires_accompanying_claim`, `labelling_requirement`, `compound_condition`,
  `cross_reference`, `requires_npsc`). This absorbed what would have been two
  near-empty auxiliary tables.
- **`npsc_scoring_reference`** — Schedule 5's step-function lookup tables (Tables
  1-5) used to *compute* a product's NPSC score. Genuinely different data shape from
  `claim_rules` (a ladder of point brackets feeding a formula, not a pass/fail
  condition per property), so kept separate rather than forced in.
- **`npsc_category_thresholds`** — Schedule 4 S4-6's three pass/fail lines
  (Category 1/2/3) that a computed NPSC score is checked against. Tiny (3 rows) but
  structurally distinct from both of the above (it's neither a claim rule nor a
  scoring ladder — it's the category definition + cutoff the ladder's output is
  compared to).

The blanket "no claims on infant formula" rule is recorded in `claim_rules` for
traceability (`claim_category='general_restriction'`) but the actual per-product
check reuses the existing `out_of_scope` tag in `product_nppm_category_mapping.db`
rather than duplicating a product list here.

## `claim_rules` schema

One row per rule. `property`/`descriptor` follow Schedule 4's own Column 1/Column 3
naming (descriptor synonym groups like "reduced/light/lite" are stored as one
combined value `reduced_light_lite`, matching how Schedule 4's own table groups
them — not three duplicate rows). `population_scope` is `general` (Schedule 4) or
`food_for_infants` (Standard 2.9.2 override) — where both exist for the same
property/descriptor, the `food_for_infants` row's `notes` field says explicitly that
it overrides the general row. `alt_group` marks rows that are OR-alternatives (any
one satisfying condition is sufficient) rather than AND-combined — e.g. the three
paths to a "diet" energy claim. **`computable_with_current_data`** (0/1) is an honest
flag: `products.db` currently tracks only 8 core nutrients (energy, protein, fat,
saturated fat, carbohydrate, sugars, sodium, dietary fibre) — 93 of the 152 rows
reference a property/nutrient outside that set (cholesterol, gluten, glycaemic
index/load, lactose, individual fatty-acid families, most vitamins/minerals) and
cannot currently be checked against real product data. They are still recorded, both
for legal completeness/citation and so the app can show "cannot verify — outside our
current data" rather than silently omitting the rule.

## `npsc_scoring_reference` schema

`points_type` is one of the five scoring components; `category_group` distinguishes
Schedule 5's Table 1 (Category 1/2 foods) from Table 2 (Category 3 foods) for the
four baseline components — V/P/F points use the same table (3/4/5) regardless of
category. Each row is one bracket: `operator`/`threshold_value`/`threshold_unit`
describe the bracket boundary exactly as printed (the very first bracket of each
ladder uses `<=`, every subsequent bracket uses `>`, `>=` or `=` exactly as Schedule 5
states it — not force-normalised to one operator). To score a product: for each
component, find the highest-points bracket whose condition the product's actual
value satisfies.

## `npsc_category_thresholds` schema

Three rows (Category 1 Beverages / Category 2 general food / Category 3 cheese &
edible oils), each with the category's defining rule and the score a product's
computed NPSC (via `npsc_scoring_reference`) must fall *below* to "meet the NPSC".

## Known gaps, honestly flagged (do not silently build around these)

1. **NPSC V points need a fruit/veg/nut/legume percentage-by-weight figure that
   `products.db` does not currently have** — OFF's `categories_tags` can approximate
   category membership but not the precise % the Schedule 5 formula needs. Any
   `requires_npsc` health claim is therefore currently only partially computable:
   baseline/P/F points can be computed from existing 8-nutrient data, V points cannot.
2. **Comparative claims (`requires_reference_food=1`) need a brand-declared reference
   food that does not exist anywhere in `products.db` or Open Food Facts** — same
   structural gap already flagged for `reference_values.db`. These will likely remain
   permanently "cannot verify" rather than resolvable.
3. **93 of 152 `claim_rules` rows reference nutrients outside the app's current
   8-nutrient scope** (see `computable_with_current_data` above) — mostly vitamins,
   minerals, and specific fatty-acid families. Recorded for completeness; not
   currently checkable against real product rows.
4. **The Category 1/2/3 split used by NPSC (Schedule 4 S4-6) is a separate
   classification from the 13-category WHO NPPM system already built for Epic 3** —
   the two must not be conflated. A small additional classification step (beverage
   tag → Category 1; cheese with >320mg calcium/100g or oil/spread/margarine/butter
   tags → Category 3; else Category 2) is needed before NPSC can be computed for a
   real product row — not yet built, flagged as a next step below.
5. `getting-your-claims-right.pdf` is current only to Amendment No. 174 (Feb 2018),
   materially behind Schedule 4's current Amendment No. 249 — used here only to
   inform the claim-verification decision-tree/UX flow, never as a source of current
   numeric thresholds.

## Suggested backend verification flow (not implemented here — this notebook builds
data only, no application logic)

1. Product classified `out_of_scope` in `product_nppm_category_mapping.db` (infant
   formula) → any claim is automatically "Not Supported", stop.
2. Determine whether the product meets Standard 2.9.2's "food for infants" definition
   (narrower than the app's overall product scope) → prefer a
   `population_scope='food_for_infants'` `claim_rules` row over `general` when both
   exist for the same property/descriptor.
3. Match the claim to a `claim_rules` row by `property`+`descriptor`.
4. If `rule_type='numeric_threshold'` and `computable_with_current_data=1`: compare
   the product's real per-100g value (from `products.db`) against the rule.
5. If `requires_npsc=1`: compute the product's NPSC score via
   `npsc_scoring_reference` (V points currently unavailable — see gap 1), classify
   into a Category 1/2/3 (see gap 4), compare against `npsc_category_thresholds`.
6. If `requires_reference_food=1`: "cannot verify" (gap 2).
7. If `computable_with_current_data=0`: "cannot verify — outside current data scope"
   (gap 3).
8. If `rule_type` is `not_permitted` or `prohibited_representation`: automatic "Not
   Supported" regardless of numeric data.


## Build code

Executed exactly as shown below (`build_claim_rules.py`), real captured output in
the cell beneath it — nothing here is a fabricated preview.

In [1]:
"""
Epic 4 -- Food Trust -- claim_rules.db builder

Builds three tables from five source PDFs (all in claim_data/):
  1. claim_rules              -- one row per claim-eligibility rule (nutrition content
                                  claims, health claims, and cross-cutting restrictions)
  2. npsc_scoring_reference    -- Schedule 5's step-tables for computing a product's
                                  Nutrient Profiling Scoring Criterion (NPSC) score
  3. npsc_category_thresholds  -- Schedule 4 S4-6's three category pass/fail lines that
                                  an NPSC score computed via table 2 is checked against

Sources (all already read page-by-page this session, transcribed here, nothing invented):
  - Schedule 4 -- Nutrition, health and related claims (F2026C00415, current 30 Apr 2026)
  - Standard 1.2.7 -- Nutrition, health and related claims (F2025C00833, current 13 Aug 2025)
  - Schedule 5 -- Nutrient profiling scoring method (F2022C01053, current 1 Sep 2022)
  - Standard 2.9.2 -- Food for infants (F2024C00903, current 13 Sep 2024)
  - "Getting Your Claims Right" guide (Feb 2018) -- informs notes only, not a legal source

Design notes (agreed with user 2026-08-29):
  - ONE table (claim_rules) for both numeric-threshold rules and non-numeric
    absolute-prohibition rules, discriminated by `rule_type` -- collapsed from an
    earlier 3-table proposal the user correctly pushed back on as over-split.
  - The infant-formula blanket ban is NOT stored as its own product-list table --
    it is documented here as a claim_rules row for traceability, but the actual
    per-product check reuses product_nppm_category_mapping.db's existing
    `out_of_scope` tag rather than duplicating that data.
  - npsc_scoring_reference / npsc_category_thresholds ARE kept separate from
    claim_rules because they are a genuinely different data shape (a step-function
    lookup table feeding a score calculation, not a pass/fail eligibility rule) --
    this is not the same mistake as the original 3-way split.
"""
import sqlite3
import os

DB_PATH = os.path.join(os.path.dirname(__file__), "claim_rules.db")
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# ---------------------------------------------------------------------------
# Table 1: claim_rules
# ---------------------------------------------------------------------------
cur.execute("""
CREATE TABLE claim_rules (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    claim_category TEXT NOT NULL,      -- nutrition_content | health_high | health_general | general_restriction
    property TEXT,                     -- e.g. sodium, protein, calcium; NULL for whole-standard restrictions
    descriptor TEXT,                   -- e.g. low, reduced_light_lite, good_source; NULL for baseline/general condition rows
    population_scope TEXT NOT NULL,    -- general | food_for_infants
    rule_type TEXT NOT NULL,           -- numeric_threshold | not_permitted | prohibited_representation |
                                        -- requires_accompanying_claim | labelling_requirement | compound_condition | cross_reference
    condition TEXT,                    -- <=, >=, =, no_detectable, range, not_permitted, see notes
    value REAL,
    value_max REAL,                    -- only used for range conditions (e.g. medium GI 56-69)
    unit TEXT,
    basis TEXT,                        -- per_100g | per_100mL | per_serving | percent_energy | percent_total_fatty_acids | percent_RDI | per_day
    product_form TEXT,                 -- solid | liquid | NULL (distinguishes dual per_100g/per_100mL rows)
    requires_npsc INTEGER DEFAULT 0,
    requires_reference_food INTEGER DEFAULT 0,
    alt_group TEXT,                    -- rows sharing (alt_group, property, descriptor, population_scope) are OR-alternatives
    relevant_population TEXT,          -- e.g. "women of childbearing age", "persons 65+", "children"
    health_effect TEXT,                -- for health claims only
    excluded_food_types TEXT,
    computable_with_current_data INTEGER DEFAULT 1,  -- 0 = needs a nutrient/field products.db does not have
    notes TEXT,
    source TEXT NOT NULL,
    source_section TEXT NOT NULL
)
""")

def r(**kw):
    row = dict(
        claim_category=None, property=None, descriptor=None, population_scope="general",
        rule_type=None, condition=None, value=None, value_max=None, unit=None, basis=None,
        product_form=None, requires_npsc=0, requires_reference_food=0, alt_group=None,
        relevant_population=None, health_effect=None, excluded_food_types=None,
        computable_with_current_data=1, notes=None, source=None, source_section=None,
    )
    row.update(kw)
    return row

ROWS = []

# ===========================================================================
# A. GENERAL RESTRICTIONS -- apply across the whole standard, not to one property
# ===========================================================================
ROWS += [
    r(claim_category="general_restriction", rule_type="not_permitted", population_scope="general",
      notes="No nutrition content claim or health claim may be made about kava.",
      source="Standard 1.2.7", source_section="1.2.7-4(1)(a)"),
    r(claim_category="general_restriction", rule_type="not_permitted", population_scope="general",
      notes="No nutrition content claim or health claim may be made about an infant formula product. "
            "Application layer: reuse product_nppm_category_mapping.db's existing out_of_scope tag "
            "instead of a separate product list -- this row exists for legal traceability only.",
      source="Standard 1.2.7", source_section="1.2.7-4(1)(b)"),
    r(claim_category="general_restriction", rule_type="not_permitted", population_scope="general",
      notes="For food containing more than 1.15% alcohol by volume: no nutrition content or health claim "
            "permitted EXCEPT nutrition content claims about salt/sodium content (non-beverage food only), "
            "carbohydrate content, energy content, gluten content, or sugar/sugars content. A nutrition "
            "content claim about such food must also not name/refer to a carbohydrate component other than "
            "sugar/sugars, or an individually named sugar (e.g. fructose).",
      source="Standard 1.2.7", source_section="1.2.7-4(1)(c), 1.2.7-4(2)"),
    r(claim_category="general_restriction", rule_type="not_permitted", population_scope="general",
      notes="A claim must not refer to the prevention, diagnosis, cure or alleviation of a disease, disorder "
            "or condition, and must not compare a food with a good that is (or is likely to be taken as) for "
            "therapeutic use.",
      source="Standard 1.2.7", source_section="1.2.7-8"),
    r(claim_category="general_restriction", property="vitamin_or_mineral", rule_type="not_permitted",
      population_scope="general",
      notes="A claim must not directly or indirectly compare the vitamin or mineral content of a food with "
            "that of another food, unless expressly permitted elsewhere in the Code.",
      source="Standard 1.2.7", source_section="1.2.7-9"),
    r(claim_category="general_restriction", property="vitamin_or_mineral", rule_type="not_permitted",
      population_scope="food_for_infants",
      notes="A claim must not compare the vitamin or mineral content of a food for infants with that of "
            "any other food, unless expressly permitted elsewhere in the Code.",
      source="Standard 2.9.2", source_section="2.9.2-10(1)"),
    r(claim_category="general_restriction", rule_type="compound_condition", population_scope="general",
      notes="For a property NOT listed in S4-3 Column 1: the claim may only state presence/absence of the "
            "property, a specified amount of the property, or a combination of both -- comparative claims "
            "(reduced/increased etc.) are barred entirely for non-S4-3 properties, and a statement of absence "
            "must not use an S4-3 Column 3 descriptor (or any other descriptor) except one that itself "
            "indicates absence.",
      source="Standard 1.2.7", source_section="1.2.7-13"),
    r(claim_category="general_restriction", property="choline_fluoride_folic_acid", rule_type="requires_accompanying_claim",
      population_scope="general",
      notes="A nutrition content claim about choline, fluoride or folic acid may state only that the food "
            "contains the substance, or contains a specified amount, or a combination -- no descriptor "
            "(high/low/rich/source of, etc.) may be used -- AND the nutrition content claim may only be made "
            "if a health claim about the same substance is also made about the same food.",
      source="Standard 1.2.7", source_section="1.2.7-14"),
    r(claim_category="general_restriction", rule_type="not_permitted", population_scope="general",
      notes="A nutrition content claim that meets the conditions for the descriptor 'diet' must not also use "
            "another descriptor that directly or indirectly refers to slimming, or a synonym for slimming.",
      source="Standard 1.2.7", source_section="1.2.7-15"),
    r(claim_category="general_restriction", rule_type="compound_condition", population_scope="general",
      notes="Comparative claim (a claim that directly/indirectly compares nutrition content between foods or "
            "brands using light/lite, increased, reduced or words of similar import; or uses 'diet' where the "
            "food has >=40% less energy than the reference food) must state, together with the claim, the "
            "identity of the reference food AND the difference in the amount of the property between the "
            "claimed food and the reference food.",
      source="Standard 1.2.7", source_section="1.2.7-16"),
    r(claim_category="general_restriction", rule_type="prohibited_representation", population_scope="food_for_infants",
      notes="A food must not be represented as being the sole or principal source of nutrition for infants.",
      source="Standard 2.9.2", source_section="2.9.2-9(1)"),
    r(claim_category="general_restriction", rule_type="prohibited_representation", population_scope="food_for_infants",
      notes="The label on a package of food for infants must not include a recommendation that the food can "
            "be added to bottle feeds of an infant formula product.",
      source="Standard 2.9.2", source_section="2.9.2-9(2)"),
    r(claim_category="general_restriction", rule_type="labelling_requirement", population_scope="general",
      notes="If a nutrition content or health claim is made, a Nutrition Information Panel (Standard 1.2.8) "
            "must be displayed/provided even for foods that would otherwise be NIP-exempt (small packages "
            "retain a partial exemption but must still carry certain claim-related labelling).",
      source="Standard 1.2.7 / Standard 1.2.8", source_section="1.2.8-6, referenced via Getting Your Claims Right s2.5"),
]

# ===========================================================================
# B. NUTRITION CONTENT CLAIMS -- Schedule 4 S4-3, population_scope='general'
#    unless a Standard 2.9.2 override row is given immediately after.
# ===========================================================================

# --- Carbohydrate ---
ROWS += [
    r(claim_category="nutrition_content", property="carbohydrate", descriptor="reduced_light_lite",
      rule_type="numeric_threshold", condition=">=", value=25, unit="percent_less", basis="relative_to_reference",
      requires_reference_food=1, source="Schedule 4", source_section="S4-3 Carbohydrate"),
    r(claim_category="nutrition_content", property="carbohydrate", descriptor="increased",
      rule_type="numeric_threshold", condition=">=", value=25, unit="percent_more", basis="relative_to_reference",
      requires_reference_food=1, source="Schedule 4", source_section="S4-3 Carbohydrate"),
]

# --- Cholesterol (not in products.db -- flagged not computable) ---
ROWS += [
    r(claim_category="nutrition_content", property="cholesterol", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=10, unit="mg", basis="per_100mL", product_form="liquid",
      computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Cholesterol"),
    r(claim_category="nutrition_content", property="cholesterol", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=20, unit="mg", basis="per_100g", product_form="solid",
      computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Cholesterol"),
    r(claim_category="nutrition_content", property="cholesterol", descriptor="reduced_light_lite",
      rule_type="numeric_threshold", condition=">=", value=25, unit="percent_less", basis="relative_to_reference",
      requires_reference_food=1, computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Cholesterol"),
]

# --- Dietary fibre ---
ROWS += [
    r(claim_category="nutrition_content", property="dietary_fibre", descriptor=None, rule_type="numeric_threshold",
      condition=">=", value=2, unit="g", basis="per_serving",
      notes="General baseline condition -- applies whenever dietary fibre is claimed, unless a low/reduced claim is being made.",
      source="Schedule 4", source_section="S4-3 Dietary fibre"),
    r(claim_category="nutrition_content", property="dietary_fibre", descriptor="good_source", rule_type="numeric_threshold",
      condition=">=", value=4, unit="g", basis="per_serving", source="Schedule 4", source_section="S4-3 Dietary fibre"),
    r(claim_category="nutrition_content", property="dietary_fibre", descriptor="excellent_source", rule_type="numeric_threshold",
      condition=">=", value=7, unit="g", basis="per_serving", source="Schedule 4", source_section="S4-3 Dietary fibre"),
    r(claim_category="nutrition_content", property="dietary_fibre", descriptor="increased", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_more", basis="relative_to_reference", requires_reference_food=1,
      notes="Reference food must itself contain >=2g dietary fibre per serving.",
      source="Schedule 4", source_section="S4-3 Dietary fibre"),
]

# --- Energy ---
ROWS += [
    r(claim_category="nutrition_content", property="energy", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=80, unit="kJ", basis="per_100mL", product_form="liquid", source="Schedule 4", source_section="S4-3 Energy"),
    r(claim_category="nutrition_content", property="energy", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=170, unit="kJ", basis="per_100g", product_form="solid", source="Schedule 4", source_section="S4-3 Energy"),
    r(claim_category="nutrition_content", property="energy", descriptor="reduced_light_lite", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_less", basis="relative_to_reference", requires_reference_food=1,
      source="Schedule 4", source_section="S4-3 Energy"),
    r(claim_category="nutrition_content", property="energy", descriptor="diet", rule_type="requires_npsc",
      requires_npsc=1, alt_group="diet_energy",
      notes="Path 1 of 2 (any one path is sufficient): food meets the NPSC (unless it is a special purpose food).",
      source="Schedule 4", source_section="S4-3 Energy"),
    r(claim_category="nutrition_content", property="energy", descriptor="diet", rule_type="numeric_threshold",
      condition="<=", value=80, unit="kJ", basis="per_100mL", product_form="liquid", alt_group="diet_energy",
      notes="Path 2 of 2, liquid form -- only available if food is a special purpose food.",
      source="Schedule 4", source_section="S4-3 Energy"),
    r(claim_category="nutrition_content", property="energy", descriptor="diet", rule_type="numeric_threshold",
      condition="<=", value=170, unit="kJ", basis="per_100g", product_form="solid", alt_group="diet_energy",
      notes="Path 2 of 2, solid form -- only available if food is a special purpose food.",
      source="Schedule 4", source_section="S4-3 Energy"),
    r(claim_category="nutrition_content", property="energy", descriptor="diet", rule_type="numeric_threshold",
      condition=">=", value=40, unit="percent_less", basis="relative_to_reference", requires_reference_food=1,
      alt_group="diet_energy",
      notes="Path 2 of 2, alternative form -- only available if food is a special purpose food.",
      source="Schedule 4", source_section="S4-3 Energy"),
]

# --- Fat ---
ROWS += [
    r(claim_category="nutrition_content", property="fat", descriptor="percent_free", rule_type="cross_reference",
      condition="meets_low_conditions", notes="Food must meet the 'low fat' conditions below.",
      source="Schedule 4", source_section="S4-3 Fat"),
    r(claim_category="nutrition_content", property="fat", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=1.5, unit="g", basis="per_100mL", product_form="liquid", source="Schedule 4", source_section="S4-3 Fat"),
    r(claim_category="nutrition_content", property="fat", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=3, unit="g", basis="per_100g", product_form="solid", source="Schedule 4", source_section="S4-3 Fat"),
    r(claim_category="nutrition_content", property="fat", descriptor="reduced_light_lite", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_less", basis="relative_to_reference", requires_reference_food=1,
      source="Schedule 4", source_section="S4-3 Fat"),
]

# --- Gluten (not in products.db -- flagged not computable) ---
ROWS += [
    r(claim_category="nutrition_content", property="gluten", descriptor="free", rule_type="compound_condition",
      condition="no_detectable_gluten_OR_qualifying_oats_or_malted_cereal",
      notes="No detectable gluten; OR the food consists of/contains oats or oat products where gluten has not "
            "been added; OR the food consists of/contains cereals containing gluten that have been malted, "
            "where gluten has not been added and the average gluten content does not exceed 20mg/100g.",
      computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Gluten"),
    r(claim_category="nutrition_content", property="gluten", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=20, unit="mg", basis="per_100g", computable_with_current_data=0,
      source="Schedule 4", source_section="S4-3 Gluten"),
    r(claim_category="nutrition_content", property="gluten", descriptor=None, rule_type="labelling_requirement",
      notes="A gluten claim may also simply state the food 'contains gluten' or 'is high in gluten' without "
            "using the Column 3 descriptor structure.",
      computable_with_current_data=0, source="Standard 1.2.7", source_section="1.2.7-12(7)"),
]

# --- Glycaemic Index / Glycaemic load (not in products.db) ---
ROWS += [
    r(claim_category="nutrition_content", property="glycaemic_index", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=55, unit="GI_units", computable_with_current_data=0, source="Schedule 4", source_section="S4-3 GI"),
    r(claim_category="nutrition_content", property="glycaemic_index", descriptor="medium", rule_type="numeric_threshold",
      condition="range", value=56, value_max=69, unit="GI_units", computable_with_current_data=0,
      source="Schedule 4", source_section="S4-3 GI"),
    r(claim_category="nutrition_content", property="glycaemic_index", descriptor="high", rule_type="numeric_threshold",
      condition=">=", value=70, unit="GI_units", computable_with_current_data=0, source="Schedule 4", source_section="S4-3 GI"),
    r(claim_category="nutrition_content", property="glycaemic_index", descriptor=None, rule_type="requires_npsc",
      requires_npsc=1, computable_with_current_data=0,
      notes="Any GI claim requires the food to meet the NPSC unless it is a special purpose food AND the "
            "numerical GI value is stated (in the claim or the NIP).",
      source="Standard 1.2.7", source_section="1.2.7-12(6)"),
    r(claim_category="nutrition_content", property="glycaemic_load", descriptor=None, rule_type="requires_npsc",
      requires_npsc=1, computable_with_current_data=0,
      notes="Meets the NPSC unless the food is a special purpose food. Descriptors, if used, must be expressed "
            "as a number or numeric form.",
      source="Schedule 4 / Standard 1.2.7", source_section="S4-3 Glycaemic load, 1.2.7-12(6)"),
]

# --- Lactose (not in products.db) ---
ROWS += [
    r(claim_category="nutrition_content", property="lactose", descriptor="free", rule_type="numeric_threshold",
      condition="no_detectable", computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Lactose"),
    r(claim_category="nutrition_content", property="lactose", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=2, unit="g", basis="per_100g", computable_with_current_data=0,
      source="Schedule 4", source_section="S4-3 Lactose"),
]

# --- Fatty acid subtypes (not in products.db -- only total fat/saturated_fat are tracked) ---
for fam, threshold_pct in [("monounsaturated_fat", 40), ("omega_6_fat", 40), ("omega_9_fat", 40), ("polyunsaturated_fat", 40)]:
    ROWS.append(r(claim_category="nutrition_content", property=fam, descriptor=None, rule_type="compound_condition",
                   condition="general_conditions", computable_with_current_data=0,
                   notes=f"<=28% saturated+trans of total fatty acid content AND >={threshold_pct}% {fam.replace('_',' ')} of total fatty acid content.",
                   source="Schedule 4", source_section=f"S4-3 {fam.replace('_',' ').title()}"))
    ROWS.append(r(claim_category="nutrition_content", property=fam, descriptor="increased", rule_type="numeric_threshold",
                   condition=">=", value=25, unit="percent_more", basis="relative_to_reference", requires_reference_food=1,
                   computable_with_current_data=0,
                   notes="Reference food must itself meet the general conditions for this fatty acid family.",
                   source="Schedule 4", source_section=f"S4-3 {fam.replace('_',' ').title()}"))

ROWS += [
    r(claim_category="nutrition_content", property="omega_3_fat", descriptor="good_source", rule_type="compound_condition",
      alt_group="omega3_good_source", condition=">= 60mg EPA+DHA per serving", computable_with_current_data=0,
      notes="Alternative A of 2: >=60mg combined EPA+DHA per serving.", source="Schedule 4", source_section="S4-3 Omega-3"),
    r(claim_category="nutrition_content", property="omega_3_fat", descriptor="good_source", rule_type="compound_condition",
      alt_group="omega3_good_source", condition=">= 200mg alpha-linolenic acid per serving", computable_with_current_data=0,
      notes="Alternative B of 2: >=200mg alpha-linolenic acid per serving. For non-fish products, ALSO requires "
            "either <=28% saturated+trans of total fatty acid content, or <=5g/100g saturated+trans fatty acids.",
      source="Schedule 4", source_section="S4-3 Omega-3"),
    r(claim_category="nutrition_content", property="omega_3_fat", descriptor="increased", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_more", basis="relative_to_reference", requires_reference_food=1,
      computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Omega-3"),
    r(claim_category="nutrition_content", property="omega_fat_any", descriptor=None, rule_type="labelling_requirement",
      computable_with_current_data=0,
      notes="Whichever omega type is claimed (3/6/9/any) must be specified immediately after the word 'omega'.",
      source="Schedule 4", source_section="S4-3 Omega fatty acids (any)"),
]

# --- Potassium (labelling only, not a numeric claim gate; not in products.db) ---
ROWS += [
    r(claim_category="nutrition_content", property="potassium", descriptor=None, rule_type="labelling_requirement",
      computable_with_current_data=0,
      notes="No standalone numeric claim condition; the Nutrition Information Panel must indicate sodium and "
            "potassium content together if potassium is discussed.",
      source="Schedule 4", source_section="S4-3 Potassium"),
]

# --- Protein ---
ROWS += [
    r(claim_category="nutrition_content", property="protein", descriptor=None, rule_type="numeric_threshold",
      condition=">=", value=5, unit="g", basis="per_serving",
      notes="General baseline condition -- applies unless a low/reduced claim is being made.",
      source="Schedule 4", source_section="S4-3 Protein"),
    r(claim_category="nutrition_content", property="protein", descriptor="good_source", rule_type="numeric_threshold",
      condition=">=", value=10, unit="g", basis="per_serving", source="Schedule 4", source_section="S4-3 Protein"),
    r(claim_category="nutrition_content", property="protein", descriptor="increased", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_more", basis="relative_to_reference", requires_reference_food=1,
      notes="Reference food must itself meet the general protein claim condition.",
      source="Schedule 4", source_section="S4-3 Protein"),
    # Standard 2.9.2 override for food for infants
    r(claim_category="nutrition_content", property="protein", descriptor="source_of",
      population_scope="food_for_infants", rule_type="numeric_threshold",
      condition=">=", value=12, unit="percent", basis="percent_energy",
      notes="OVERRIDES the general baseline row above for food that meets Standard 2.9.2's 'food for infants' "
            "definition. A claim must not be made that a food for infants is a source of protein unless at "
            "least 12% of the average energy content of the food is derived from protein.",
      source="Standard 2.9.2", source_section="2.9.2-8(2)"),
]

# --- Salt or sodium ---
ROWS += [
    r(claim_category="nutrition_content", property="sodium", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=120, unit="mg", basis="per_100mL", product_form="liquid", source="Schedule 4", source_section="S4-3 Salt or sodium"),
    r(claim_category="nutrition_content", property="sodium", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=120, unit="mg", basis="per_100g", product_form="solid", source="Schedule 4", source_section="S4-3 Salt or sodium"),
    r(claim_category="nutrition_content", property="sodium", descriptor="reduced_light_lite", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_less", basis="relative_to_reference", requires_reference_food=1,
      source="Schedule 4", source_section="S4-3 Salt or sodium"),
    r(claim_category="nutrition_content", property="sodium", descriptor="no_added", rule_type="compound_condition",
      condition="no_added_sodium_compound",
      notes="(a) The food for sale contains no added sodium compound (including no added salt); AND "
            "(b) none of the ingredients used to make the food contains an added sodium compound (including no added salt).",
      source="Schedule 4", source_section="S4-3 Salt or sodium"),
    r(claim_category="nutrition_content", property="sodium", descriptor="unsalted", rule_type="cross_reference",
      condition="meets_no_added_conditions", notes="Food must meet the 'no added salt or sodium' conditions above.",
      source="Schedule 4", source_section="S4-3 Salt or sodium"),
]

# --- Saturated and trans fatty acids (combined descriptor) ---
ROWS += [
    r(claim_category="nutrition_content", property="saturated_and_trans_fat", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=0.75, unit="g", basis="per_100mL", product_form="liquid",
      computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Saturated and trans fatty acids"),
    r(claim_category="nutrition_content", property="saturated_and_trans_fat", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=1.5, unit="g", basis="per_100g", product_form="solid",
      computable_with_current_data=0, source="Schedule 4", source_section="S4-3 Saturated and trans fatty acids"),
    r(claim_category="nutrition_content", property="saturated_and_trans_fat", descriptor="reduced_light_lite",
      rule_type="compound_condition", requires_reference_food=1, computable_with_current_data=0,
      notes=">=25% less than the reference food AND both the saturated and trans fatty acid content are "
            "reduced relative to the reference food.",
      source="Schedule 4", source_section="S4-3 Saturated and trans fatty acids"),
    r(claim_category="nutrition_content", property="saturated_and_trans_fat", descriptor="low_proportion",
      rule_type="compound_condition", computable_with_current_data=0,
      notes="<=28% of total fatty acid content, AND the claim must expressly state 'low proportion of "
            "saturated and trans fatty acids of total fatty acid content'.",
      source="Schedule 4", source_section="S4-3 Saturated and trans fatty acids"),
]

# --- Saturated fatty acids (alone) ---
ROWS += [
    r(claim_category="nutrition_content", property="saturated_fat", descriptor="free", rule_type="compound_condition",
      notes="No detectable saturated fatty acids AND no detectable trans fatty acids.",
      source="Schedule 4", source_section="S4-3 Saturated fatty acids"),
    r(claim_category="nutrition_content", property="saturated_fat", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=0.75, unit="g", basis="per_100mL", product_form="liquid", source="Schedule 4", source_section="S4-3 Saturated fatty acids"),
    r(claim_category="nutrition_content", property="saturated_fat", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=1.5, unit="g", basis="per_100g", product_form="solid", source="Schedule 4", source_section="S4-3 Saturated fatty acids"),
    r(claim_category="nutrition_content", property="saturated_fat", descriptor="reduced_light_lite",
      rule_type="compound_condition", requires_reference_food=1,
      notes=">=25% less saturated fatty acids than the reference food AND no more trans fatty acids than the reference food.",
      source="Schedule 4", source_section="S4-3 Saturated fatty acids"),
    r(claim_category="nutrition_content", property="saturated_fat", descriptor="low_proportion", rule_type="compound_condition",
      notes="<=28% of total fatty acid content, AND the claim must expressly state 'low proportion of "
            "saturated fatty acids of total fatty acid content'.",
      source="Schedule 4", source_section="S4-3 Saturated fatty acids"),
]

# --- Sugar or sugars ---
ROWS += [
    r(claim_category="nutrition_content", property="sugars", descriptor="percent_free", rule_type="cross_reference",
      condition="meets_low_conditions", notes="Food must meet the 'low sugar' conditions below.",
      source="Schedule 4", source_section="S4-3 Sugar or sugars"),
    r(claim_category="nutrition_content", property="sugars", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=2.5, unit="g", basis="per_100mL", product_form="liquid", source="Schedule 4", source_section="S4-3 Sugar or sugars"),
    r(claim_category="nutrition_content", property="sugars", descriptor="low", rule_type="numeric_threshold",
      condition="<=", value=5, unit="g", basis="per_100g", product_form="solid", source="Schedule 4", source_section="S4-3 Sugar or sugars"),
    r(claim_category="nutrition_content", property="sugars", descriptor="reduced_light_lite", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent_less", basis="relative_to_reference", requires_reference_food=1,
      source="Schedule 4", source_section="S4-3 Sugar or sugars"),
    r(claim_category="nutrition_content", property="sugars", descriptor="no_added", rule_type="compound_condition",
      condition="8_part_test", computable_with_current_data=0,
      notes="(a) The food for sale is not itself 'an added sugar'; (b) does not contain an added sugar as an "
            "added ingredient; (c) does not contain more than 10g/100g (solid) or 7.5g/100mL (liquid) sugars; "
            "(d) has not had its hexose monosaccharide/disaccharide concentration increased by hydrolysis "
            "during production; (e) condition (d) is waived if the concentration increase is not more than "
            "1.5%; (f) 'added sugar' is defined as any of 14 listed sources (hexose monosaccharides other than "
            "D-allulose and disaccharides; D-tagatose; starch hydrolysate; glucose syrup; maltodextrin and "
            "similar; sugar-refinery products incl. brown sugar/molasses/raw sugar/golden syrup/treacle; icing "
            "sugar; invert sugar; sugar/sugar syrup derived from plants; honey; native bee honey; malt; malt "
            "extracts; and, since Amendment 249 (30 Apr 2026), concentrated/deionised fruit or vegetable juice "
            "UNLESS the food is a 'prescribed beverage' -- a brewed soft drink, formulated beverage, juice "
            "blend, fruit drink, fruit juice, vegetable juice, or water-based beverage); (g) 'ingredient' "
            "includes an ingredient of a compound ingredient.",
      source="Schedule 4", source_section="S4-3 Sugar or sugars, S4-2 definitions"),
    r(claim_category="nutrition_content", property="sugars", descriptor="unsweetened", rule_type="compound_condition",
      computable_with_current_data=0,
      notes="Meets the 'no added sugar' conditions above AND does not contain intense sweeteners, sorbitol, "
            "mannitol, glycerol, xylitol, isomalt, maltitol, maltitol syrup, erythritol or lactitol AND does "
            "not contain a monosaccharide/disaccharide listed in the table to subsection S11-2(3) as an "
            "ingredient or an ingredient of a compound ingredient.",
      source="Schedule 4", source_section="S4-3 Sugar or sugars"),
    r(claim_category="nutrition_content", property="sugars", descriptor="sweetened", population_scope="food_for_infants",
      rule_type="labelling_requirement",
      notes="A 'sweetened' warning must be included on the label of a food for infants if sugars exceed 4g/100g.",
      source="Standard 2.9.2", source_section="2.9.2-7(3)(d)"),
]

# --- Trans fatty acids (alone, not in products.db) ---
ROWS += [
    r(claim_category="nutrition_content", property="trans_fat", descriptor="free", rule_type="compound_condition",
      computable_with_current_data=0,
      notes="No detectable trans fatty acids AND ([<=0.75g/100mL or <=1.5g/100g saturated fatty acids] OR "
            "<=28% saturated fatty acids of total fatty acid content).",
      source="Schedule 4", source_section="S4-3 Trans fatty acids"),
    r(claim_category="nutrition_content", property="trans_fat", descriptor="reduced_light_lite", rule_type="compound_condition",
      requires_reference_food=1, computable_with_current_data=0,
      notes=">=25% less trans fatty acids than the reference food AND no more saturated fatty acids than the "
            "reference food.",
      source="Schedule 4", source_section="S4-3 Trans fatty acids"),
]

# --- Vitamin or mineral (general claim conditions; most vitamins/minerals out of current data scope) ---
ROWS += [
    r(claim_category="nutrition_content", property="vitamin_or_mineral", descriptor=None, rule_type="compound_condition",
      computable_with_current_data=0,
      notes="The vitamin/mineral is listed in Column 1 of the S1-2/S1-3 table AND the food contains >=10% "
            "RDI/ESADDI per serving AND does not exceed the maximum claimable amount under 1.3.2-4/5 AND the "
            "food is NOT a formulated caffeinated beverage, food for infants, formulated meal replacement, "
            "formulated supplementary food, or formulated supplementary sports food (exception path exists if "
            "a max claimable amount applies, serving size < reference quantity, the reference quantity has "
            ">=10% RDI, and the max claimable amount is <10% RDI per serving).",
      source="Schedule 4", source_section="S4-3 Vitamin or mineral"),
    r(claim_category="nutrition_content", property="vitamin_or_mineral", descriptor="good_source", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent", basis="percent_RDI", computable_with_current_data=0,
      notes="Per serving, of RDI or ESADDI.", source="Schedule 4", source_section="S4-3 Vitamin or mineral"),
    r(claim_category="nutrition_content", property="vitamin_or_mineral", descriptor=None,
      population_scope="food_for_infants", rule_type="numeric_threshold",
      condition=">=", value=10, unit="percent", basis="percent_RDI", computable_with_current_data=0,
      notes="OVERRIDES the general row above for food that meets Standard 2.9.2's 'food for infants' "
            "definition. A claim as to presence of a vitamin or mineral may be made if a normal serving "
            "contains at least 10% RDI or ESADDI.",
      source="Standard 2.9.2", source_section="2.9.2-10(2)"),
    r(claim_category="nutrition_content", property="vitamin_or_mineral", descriptor="good_source",
      population_scope="food_for_infants", rule_type="numeric_threshold",
      condition=">=", value=25, unit="percent", basis="percent_RDI", computable_with_current_data=0,
      notes="A claim that a food for infants is a good source of a vitamin or mineral may be made if a "
            "reference quantity of the food contains at least 25% RDI or ESADDI.",
      source="Standard 2.9.2", source_section="2.9.2-10(3)"),
    r(claim_category="nutrition_content", property="vitamin_c", descriptor=None, population_scope="food_for_infants",
      rule_type="numeric_threshold", condition="<=", value=60, unit="mg", basis="per_100g",
      computable_with_current_data=0,
      notes="Compositional cap (not a claim condition per se): fruit-based food for infants must not exceed "
            "60mg/100g vitamin C if a vitamin/mineral claim is made about it.",
      source="Standard 2.9.2", source_section="2.9.2-10(4)"),
    r(claim_category="nutrition_content", property="folate", descriptor=None, population_scope="food_for_infants",
      rule_type="numeric_threshold", condition="<=", value=150, unit="microgram", basis="per_100g",
      computable_with_current_data=0,
      notes="Compositional cap (not a claim condition per se): fruit-based food for infants must not exceed "
            "150microgram/100g folate if a vitamin/mineral claim is made about it.",
      source="Standard 2.9.2", source_section="2.9.2-10(4)"),
]

# ===========================================================================
# C. HIGH LEVEL HEALTH CLAIMS -- Schedule 4 S4-4 (all require NPSC per 1.2.7-18(1)(a))
# ===========================================================================
ROWS += [
    r(claim_category="health_high", property="fruit_and_vegetables", health_effect="reduced risk of coronary heart disease",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      excluded_food_types="juice blend, fruit juice, vegetable juice, formulated beverage, mineral or spring "
                           "water, non-alcoholic beverage, brewed soft drink, fruit drink, electrolyte drink or base",
      notes="Food must be >=90% fruit/vegetable by weight.", source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="beta_glucan", health_effect="reduced blood cholesterol",
      rule_type="numeric_threshold", condition=">=", value=1, unit="g", basis="per_serving", requires_npsc=1,
      computable_with_current_data=0,
      notes="From oat bran, wholegrain oats or wholegrain barley. Dietary context statement: '3g beta-glucan/day'.",
      source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="calcium", health_effect="bone mineral density",
      rule_type="numeric_threshold", condition=">=", value=200, unit="mg", basis="per_serving", requires_npsc=1,
      computable_with_current_data=0, source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="calcium", health_effect="reduced risk of osteoporosis",
      relevant_population="persons 65+", rule_type="numeric_threshold", condition=">=", value=290, unit="mg",
      basis="per_serving", requires_npsc=1, computable_with_current_data=0,
      notes="Food must also be an adequate source of vitamin D.", source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="calcium_and_vitamin_d", health_effect="reduced risk of osteoporosis",
      relevant_population="persons 65+", rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      notes=">=290mg calcium/serving AND meets the general vitamin D claim conditions.",
      source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="folic_acid", health_effect="reduced risk of fetal neural tube defects",
      relevant_population="women of childbearing age", rule_type="numeric_threshold", condition=">=", value=40,
      unit="microgram", basis="per_serving", requires_npsc=1, computable_with_current_data=0,
      excluded_food_types="soft cheese, pate, liver or liver product, food containing added phytosterols/"
                           "phytostanols/esters, formulated caffeinated beverage, formulated supplementary "
                           "sports food, formulated meal replacement",
      source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="fruit_and_vegetables", health_effect="reduced risk of coronary heart disease (increased intake)",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      excluded_food_types="same exclusions as the general fruit/vegetables row above",
      notes="Food must be >=90% fruit/vegetable by weight.", source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="phytosterols", health_effect="reduced blood cholesterol",
      rule_type="numeric_threshold", condition=">=", value=0.8, unit="g", basis="per_serving", requires_npsc=1,
      computable_with_current_data=0,
      notes="Total plant sterol equivalents; must also meet the table-S25-2 conditions. Dietary context "
            "statement: '2g/day'.", source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="saturated_fat", health_effect="reduced total/LDL cholesterol",
      rule_type="cross_reference", requires_npsc=1,
      notes="Food must meet the 'low saturated fat' nutrition content claim conditions.",
      source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="saturated_and_trans_fat", health_effect="reduced total/LDL cholesterol",
      rule_type="cross_reference", requires_npsc=1, computable_with_current_data=0,
      notes="Food must meet the 'low saturated and trans fatty acids' nutrition content claim conditions.",
      source="Schedule 4", source_section="S4-4"),
    r(claim_category="health_high", property="sodium", health_effect="reduced blood pressure",
      rule_type="cross_reference", requires_npsc=1,
      notes="Food must meet the 'low sodium or salt' nutrition content claim conditions.",
      source="Schedule 4", source_section="S4-4"),
]

# ===========================================================================
# D. GENERAL LEVEL HEALTH CLAIMS -- Schedule 4 S4-5 (all require NPSC per 1.2.7-18(1)(a))
#    Parts 1 (minerals) and 2 (vitamins): most require only "meets general nutrition
#    content claim conditions for [nutrient]" -- stored compactly, cross-referencing
#    the S4-3 vitamin_or_mineral rows above rather than repeating the same condition.
#    All are flagged not computable -- products.db does not track vitamin/mineral fields.
# ===========================================================================
MINERALS_PART1 = ["calcium", "chromium", "copper", "iodine", "iron", "manganese", "magnesium",
                   "molybdenum", "phosphorus", "selenium", "zinc"]
for m in MINERALS_PART1:
    ROWS.append(r(claim_category="health_general", property=m, health_effect="general level health claim (see S4-5 Part 1 for exact wording)",
                   rule_type="cross_reference", requires_npsc=1, computable_with_current_data=0,
                   notes="Meets the general nutrition content claim conditions for this mineral (see S4-3 Vitamin or mineral row).",
                   source="Schedule 4", source_section="S4-5 Part 1"))
ROWS.append(r(claim_category="health_general", property="fluoride", health_effect="general level health claim (see S4-5 Part 1 for exact wording)",
               rule_type="numeric_threshold", condition=">=", value=0.6, unit="mg/L", requires_npsc=1,
               computable_with_current_data=0, source="Schedule 4", source_section="S4-5 Part 1"))
for m in ["calcium", "iron", "zinc"]:
    ROWS.append(r(claim_category="health_general", property=m, health_effect="contributes to normal growth and development / normal cognitive development",
                   relevant_population="children", rule_type="cross_reference", requires_npsc=1,
                   computable_with_current_data=0,
                   notes="Population-specific sub-row -- meets the general mineral claim conditions, health "
                         "effect wording restricted to growth/development or cognitive development for children.",
                   source="Schedule 4", source_section="S4-5 Part 1"))

VITAMINS_PART2 = ["biotin", "folate", "niacin", "pantothenic_acid", "riboflavin", "thiamin",
                   "vitamin_a", "vitamin_b6", "vitamin_b12", "vitamin_c", "vitamin_d", "vitamin_e", "vitamin_k"]
for v in VITAMINS_PART2:
    ROWS.append(r(claim_category="health_general", property=v, health_effect="general level health claim (see S4-5 Part 2 for exact wording)",
                   rule_type="cross_reference", requires_npsc=1, computable_with_current_data=0,
                   notes="Meets the general nutrition content claim conditions for this vitamin.",
                   source="Schedule 4", source_section="S4-5 Part 2"))
ROWS.append(r(claim_category="health_general", property="choline", health_effect="general level health claim (see S4-5 Part 2 for exact wording)",
               rule_type="numeric_threshold", condition=">=", value=50, unit="mg", basis="per_serving",
               requires_npsc=1, computable_with_current_data=0, source="Schedule 4", source_section="S4-5 Part 2"))
ROWS.append(r(claim_category="health_general", property="folic_acid", health_effect="reduced risk of fetal neural tube defects",
               relevant_population="women of childbearing age", rule_type="numeric_threshold", condition=">=",
               value=40, unit="microgram", basis="per_serving", requires_npsc=1, computable_with_current_data=0,
               excluded_food_types="soft cheese, pate, liver or liver product, food containing added "
                                    "phytosterols/phytostanols/esters, formulated caffeinated beverage, "
                                    "formulated supplementary sports food, formulated meal replacement",
               notes="Diet-level target referenced in the guide: >=400microgram folic acid/day.",
               source="Schedule 4", source_section="S4-5 Part 2"))
for v in ["vitamin_a", "vitamin_c", "vitamin_d"]:
    ROWS.append(r(claim_category="health_general", property=v, health_effect="general level health claim, population-specific wording",
                   relevant_population="children", rule_type="cross_reference", requires_npsc=1,
                   computable_with_current_data=0, source="Schedule 4", source_section="S4-5 Part 2"))

# --- Part 3: Other ---
ROWS += [
    r(claim_category="health_general", property="beta_glucan", health_effect="reduced blood cholesterol",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      notes="Same structure as the S4-4 high level claim: food must contain oat/barley-based beta-glucan food "
            "with >=1g beta-glucan/serving. Dietary context statement '3g/day'.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="carbohydrate", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      notes=">=55% of energy content of the food OR the food is a formulated meal or formulated supplementary "
            "food with <=10% carbohydrate content from sugars.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="carbohydrate", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      relevant_population="young children aged 1-3 years", rule_type="numeric_threshold", condition="<=",
      value=10, unit="percent", basis="percent_energy", requires_npsc=1, computable_with_current_data=0,
      notes="Population-specific sub-row -- food must be a formulated supplementary food for young children "
            "with max 10% carbohydrate content from sugars.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="dietary_fibre", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="cross_reference", requires_npsc=1,
      notes="Meets the general dietary fibre nutrition content claim conditions.", source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="epa_dha", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="numeric_threshold", condition=">=", value=50, unit="mg", basis="per_serving", requires_npsc=1,
      computable_with_current_data=0,
      notes="Combined EPA+DHA per serving. Dietary context '500mg/day'. Non-fish products also need the "
            "saturated+trans fat conditions used for Omega-3.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="energy", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="numeric_threshold", condition=">=", value=420, unit="kJ", basis="per_serving", requires_npsc=1,
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="energy", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      relevant_population="young children aged 1-3 years", rule_type="compound_condition", requires_npsc=1,
      notes="Population-specific sub-row -- food must be a formulated supplementary food for young children.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="energy", health_effect="weight loss / weight maintenance",
      rule_type="compound_condition", requires_npsc=1,
      alt_group="energy_weight",
      notes="Path 1: meets the 'diet' nutrition content claim conditions.", source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="energy", health_effect="weight loss / weight maintenance",
      rule_type="numeric_threshold", condition="<=", value=1200, unit="kJ", basis="per_serving", requires_npsc=1,
      alt_group="energy_weight",
      notes="Path 2: food is a formulated meal replacement, <=1200kJ/serving.", source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="live_yoghurt_cultures", health_effect="improved lactose digestion for lactose-intolerant individuals",
      rule_type="numeric_threshold", condition=">=", value=1e8, unit="cfu/g", requires_npsc=1,
      computable_with_current_data=0,
      notes="Yoghurt or fermented milk containing >=10^8 cfu/g of Lactobacillus delbrueckii subsp. bulgaricus "
            "and Streptococcus thermophilus.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="phytosterols", health_effect="reduced blood cholesterol",
      rule_type="numeric_threshold", condition=">=", value=0.8, unit="g", basis="per_serving", requires_npsc=1,
      computable_with_current_data=0,
      notes="Same conditions as the S4-4 high level claim.", source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="potassium", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="numeric_threshold", condition=">=", value=200, unit="mg", basis="per_serving", requires_npsc=1,
      computable_with_current_data=0, source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="protein", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="cross_reference", requires_npsc=1,
      notes="Meets the general protein nutrition content claim conditions.", source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="protein", health_effect="necessary for normal growth and development",
      relevant_population="infants aged 6 months to 12 months", population_scope="food_for_infants",
      rule_type="cross_reference", requires_npsc=1,
      notes="Food must be a food for infants and comply with subsection 2.9.2-8(2) (>=12% energy from protein).",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="protein", health_effect="general level health claim, population-specific wording",
      relevant_population="children and adolescents aged 4 years and over", rule_type="cross_reference", requires_npsc=1,
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="fruit_and_vegetables", health_effect="general level health claim (see S4-5 Part 3 for exact wording)",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      excluded_food_types="same exclusions as the S4-4 fruit/vegetables rows",
      notes="Food must be >=90% fruit/vegetable by weight.", source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="sugars", health_effect="does not promote tooth decay / dental health",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      alt_group="sugars_dental",
      notes="Path 1: confectionery or chewing gum with <=0.2% fermentable carbohydrates.",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="sugars", health_effect="does not promote tooth decay / dental health",
      rule_type="compound_condition", requires_npsc=1, computable_with_current_data=0,
      alt_group="sugars_dental",
      notes="Path 2: confectionery or chewing gum meeting an indwelling-plaque-pH test (per the 1983 Imfeld "
            "reference cited in Schedule 4).",
      source="Schedule 4", source_section="S4-5 Part 3"),
    r(claim_category="health_general", property="chewing_gum", health_effect="tooth mineralisation / plaque neutralisation / reduced oral dryness",
      rule_type="cross_reference", requires_npsc=1, computable_with_current_data=0,
      source="Schedule 4", source_section="S4-5 Part 3"),
]

cols = list(ROWS[0].keys())
placeholders = ",".join("?" for _ in cols)
cur.executemany(
    f"INSERT INTO claim_rules ({','.join(cols)}) VALUES ({placeholders})",
    [tuple(row[c] for c in cols) for row in ROWS],
)
print(f"claim_rules: inserted {len(ROWS)} rows")

# ===========================================================================
# Table 2: npsc_scoring_reference  (Schedule 5)
# ===========================================================================
cur.execute("""
CREATE TABLE npsc_scoring_reference (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    points_type TEXT NOT NULL,     -- baseline_energy | baseline_saturated_fat | baseline_sugars | baseline_sodium | v_points | p_points | f_points
    category_group TEXT NOT NULL,  -- cat1_2 | cat3 | all
    column_variant TEXT,           -- v_points only: concentrated | standard_fvnl
    points INTEGER NOT NULL,
    operator TEXT NOT NULL,        -- '<=' for the first (0-point) row, '>' for every row after
    threshold_value REAL NOT NULL,
    threshold_unit TEXT NOT NULL,
    source_table TEXT NOT NULL,
    notes TEXT
)
""")

def npsc(**kw):
    return kw

NPSC_ROWS = []

# Table 1 -- Baseline points for Category 1 or 2 foods (energy kJ, sat fat g, sugars g, sodium mg, per unit quantity)
TABLE1 = [
    (0, "<=", 335, 1.0, 5.0, 90),
    (1, ">", 335, 1.0, 5.0, 90),
    (2, ">", 670, 2.0, 9.0, 180),
    (3, ">", 1005, 3.0, 13.5, 270),
    (4, ">", 1340, 4.0, 18.0, 360),
    (5, ">", 1675, 5.0, 22.5, 450),
    (6, ">", 2010, 6.0, 27.0, 540),
    (7, ">", 2345, 7.0, 31.0, 630),
    (8, ">", 2680, 8.0, 36.0, 720),
    (9, ">", 3015, 9.0, 40.0, 810),
    (10, ">", 3350, 10.0, 45.0, 900),
]
for pts, op, energy_kj, satfat_g, sugars_g, sodium_mg in TABLE1:
    NPSC_ROWS.append(npsc(points_type="baseline_energy", category_group="cat1_2", points=pts, operator=op,
                           threshold_value=energy_kj, threshold_unit="kJ", source_table="Table 1"))
    NPSC_ROWS.append(npsc(points_type="baseline_saturated_fat", category_group="cat1_2", points=pts, operator=op,
                           threshold_value=satfat_g, threshold_unit="g", source_table="Table 1"))
    NPSC_ROWS.append(npsc(points_type="baseline_sugars", category_group="cat1_2", points=pts, operator=op,
                           threshold_value=sugars_g, threshold_unit="g", source_table="Table 1"))
    NPSC_ROWS.append(npsc(points_type="baseline_sodium", category_group="cat1_2", points=pts, operator=op,
                           threshold_value=sodium_mg, threshold_unit="mg", source_table="Table 1"))

# Table 2 -- Baseline points for Category 3 foods (same energy/satfat/sugars ladder as Table 1 up to point 10,
# but continues to point 30 for satfat/sodium; sugars ladder stops at point 10 per the source, no >45.0 row given)
TABLE2_EXTRA = [
    (11, ">", 3685, 11.0, None, 990),
    (12, ">", None, 12.0, None, 1080),
    (13, ">", None, 13.0, None, 1170),
    (14, ">", None, 14.0, None, 1260),
    (15, ">", None, 15.0, None, 1350),
    (16, ">", None, 16.0, None, 1440),
    (17, ">", None, 17.0, None, 1530),
    (18, ">", None, 18.0, None, 1620),
    (19, ">", None, 19.0, None, 1710),
    (20, ">", None, 20.0, None, 1800),
    (21, ">", None, 21.0, None, 1890),
    (22, ">", None, 22.0, None, 1980),
    (23, ">", None, 23.0, None, 2070),
    (24, ">", None, 24.0, None, 2160),
    (25, ">", None, 25.0, None, 2250),
    (26, ">", None, 26.0, None, 2340),
    (27, ">", None, 27.0, None, 2430),
    (28, ">", None, 28.0, None, 2520),
    (29, ">", None, 29.0, None, 2610),
    (30, ">", None, 30.0, None, 2700),
]
for pts, op, energy_kj, satfat_g, sugars_g, sodium_mg in TABLE1:
    NPSC_ROWS.append(npsc(points_type="baseline_energy", category_group="cat3", points=pts, operator=op,
                           threshold_value=energy_kj, threshold_unit="kJ", source_table="Table 2"))
    NPSC_ROWS.append(npsc(points_type="baseline_saturated_fat", category_group="cat3", points=pts, operator=op,
                           threshold_value=satfat_g, threshold_unit="g", source_table="Table 2"))
    NPSC_ROWS.append(npsc(points_type="baseline_sugars", category_group="cat3", points=pts, operator=op,
                           threshold_value=sugars_g, threshold_unit="g", source_table="Table 2"))
    NPSC_ROWS.append(npsc(points_type="baseline_sodium", category_group="cat3", points=pts, operator=op,
                           threshold_value=sodium_mg, threshold_unit="mg", source_table="Table 2"))
for pts, op, energy_kj, satfat_g, sugars_g, sodium_mg in TABLE2_EXTRA:
    if energy_kj is not None:
        NPSC_ROWS.append(npsc(points_type="baseline_energy", category_group="cat3", points=pts, operator=op,
                               threshold_value=energy_kj, threshold_unit="kJ", source_table="Table 2"))
    NPSC_ROWS.append(npsc(points_type="baseline_saturated_fat", category_group="cat3", points=pts, operator=op,
                           threshold_value=satfat_g, threshold_unit="g", source_table="Table 2"))
    NPSC_ROWS.append(npsc(points_type="baseline_sodium", category_group="cat3", points=pts, operator=op,
                           threshold_value=sodium_mg, threshold_unit="mg", source_table="Table 2"))

# Table 3 -- V points (fruit/vegetable/nut/legume, fvnl)
TABLE3 = [
    (0, "<", 25, 40),
    (1, ">=", 25, 41),   # Column 2 row for points 1 is "> 40" -- stored as >=41 is NOT exact; see notes below
    (2, ">=", 43, 61),
    (5, ">=", 67, 81),
    (8, "=", 100, 100),
]
# Column 1 (% concentrated fruit or vegetables) and Column 2 (% fvnl) have different operators per the source
# text (Column 1 uses >=, Column 2 uses >) -- stored faithfully per column, not force-unified.
V_COL1 = [(0, "<", 25), (1, ">=", 25), (2, ">=", 43), (5, ">=", 67), (8, "=", 100)]
V_COL2 = [(0, "<=", 40), (1, ">", 40), (2, ">", 60), (5, ">", 80), (8, "=", 100)]
for pts, op, val in V_COL1:
    NPSC_ROWS.append(npsc(points_type="v_points", category_group="all", column_variant="concentrated",
                           points=pts, operator=op, threshold_value=val, threshold_unit="percent", source_table="Table 3"))
for pts, op, val in V_COL2:
    NPSC_ROWS.append(npsc(points_type="v_points", category_group="all", column_variant="standard_fvnl",
                           points=pts, operator=op, threshold_value=val, threshold_unit="percent", source_table="Table 3"))

# Table 4 -- P points (protein g per unit quantity)
P_TABLE = [(0, "<=", 1.6), (1, ">", 1.6), (2, ">=", 3.2), (3, ">", 4.8), (4, ">", 6.4), (5, ">", 8.0)]
for pts, op, val in P_TABLE:
    NPSC_ROWS.append(npsc(points_type="p_points", category_group="all", points=pts, operator=op,
                           threshold_value=val, threshold_unit="g", source_table="Table 4",
                           notes="Foods scoring >=13 baseline points may not score protein points unless they also score >=5 V points."))

# Table 5 -- F points (dietary fibre g per unit quantity; Category 1 foods do not score F points)
F_TABLE = [(0, "<=", 0.9), (1, ">", 0.9), (2, ">", 1.9), (3, ">", 2.8), (4, ">", 3.7), (5, ">", 4.7)]
for pts, op, val in F_TABLE:
    NPSC_ROWS.append(npsc(points_type="f_points", category_group="all", points=pts, operator=op,
                           threshold_value=val, threshold_unit="g", source_table="Table 5",
                           notes="Category 1 (Beverages) foods do not score F points at all."))

npsc_cols = ["points_type", "category_group", "column_variant", "points", "operator", "threshold_value", "threshold_unit", "source_table", "notes"]
for row in NPSC_ROWS:
    for c in npsc_cols:
        row.setdefault(c, None)
placeholders = ",".join("?" for _ in npsc_cols)
cur.executemany(
    f"INSERT INTO npsc_scoring_reference ({','.join(npsc_cols)}) VALUES ({placeholders})",
    [tuple(row[c] for c in npsc_cols) for row in NPSC_ROWS],
)
print(f"npsc_scoring_reference: inserted {len(NPSC_ROWS)} rows")

# ===========================================================================
# Table 3: npsc_category_thresholds  (Schedule 4 S4-6)
# ===========================================================================
cur.execute("""
CREATE TABLE npsc_category_thresholds (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    npsc_category INTEGER NOT NULL,   -- 1, 2 or 3
    category_name TEXT NOT NULL,
    category_definition TEXT NOT NULL,
    max_score_exclusive REAL NOT NULL,  -- score must be LESS THAN this to "meet the NPSC"
    baseline_table TEXT NOT NULL,       -- which Schedule 5 table (1 or 2) supplies baseline points for this category
    source TEXT NOT NULL,
    source_section TEXT NOT NULL
)
""")
CAT_ROWS = [
    (1, "Beverages", "Beverages as defined in the Code.", 1, "Table 1", "Schedule 4", "S4-6"),
    (2, "General food", "Any food other than a food in Category 1 or Category 3. Cheese with <=320mg calcium/100g "
                         "is treated as Category 2, not Category 3.", 4, "Table 1", "Schedule 4", "S4-6"),
    (3, "Cheese and edible oils", "Cheese with >320mg calcium/100g, OR edible oil, OR edible oil spread, OR "
                                   "margarine, OR butter.", 28, "Table 2", "Schedule 4", "S4-6"),
]
cur.executemany(
    "INSERT INTO npsc_category_thresholds (npsc_category, category_name, category_definition, max_score_exclusive, baseline_table, source, source_section) VALUES (?,?,?,?,?,?,?)",
    CAT_ROWS,
)
print(f"npsc_category_thresholds: inserted {len(CAT_ROWS)} rows")

conn.commit()

# ---- verification / summary ----
for t in ["claim_rules", "npsc_scoring_reference", "npsc_category_thresholds"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows total")

print("\nclaim_rules by claim_category:")
for row in cur.execute("SELECT claim_category, COUNT(*) FROM claim_rules GROUP BY claim_category ORDER BY 2 DESC"):
    print(" ", row)

print("\nclaim_rules by population_scope:")
for row in cur.execute("SELECT population_scope, COUNT(*) FROM claim_rules GROUP BY population_scope"):
    print(" ", row)

print("\nclaim_rules by computable_with_current_data:")
for row in cur.execute("SELECT computable_with_current_data, COUNT(*) FROM claim_rules GROUP BY computable_with_current_data"):
    print(" ", row)

print("\nclaim_rules by rule_type:")
for row in cur.execute("SELECT rule_type, COUNT(*) FROM claim_rules GROUP BY rule_type ORDER BY 2 DESC"):
    print(" ", row)

conn.close()
print(f"\nBuilt {DB_PATH}")


claim_rules: inserted 152 rows
npsc_scoring_reference: inserted 151 rows
npsc_category_thresholds: inserted 3 rows
claim_rules: 152 rows total
npsc_scoring_reference: 151 rows total
npsc_category_thresholds: 3 rows total

claim_rules by claim_category:
  ('nutrition_content', 76)
  ('health_general', 52)
  ('general_restriction', 13)
  ('health_high', 11)

claim_rules by population_scope:
  ('food_for_infants', 10)
  ('general', 142)

claim_rules by computable_with_current_data:
  (0, 93)
  (1, 59)

claim_rules by rule_type:
  ('numeric_threshold', 63)
  ('cross_reference', 41)
  ('compound_condition', 30)
  ('not_permitted', 7)
  ('labelling_requirement', 5)
  ('requires_npsc', 3)
  ('prohibited_representation', 2)
  ('requires_accompanying_claim', 1)

Built /tmp/claim_build/claim_rules.db


## Next steps (not done in this build)

1. Build the small NPSC Category 1/2/3 classifier for real products (see gap 4
   above) — needed before any `requires_npsc` health claim can actually be checked.
2. Decide, with the team, whether it's worth adding a fruit/veg/nut/legume
   percentage-by-weight field to `products.db` to unblock V points (gap 1) — likely
   not derivable from Open Food Facts data at all, may need to stay a permanent gap.
3. Wire up the backend verification flow described above against `products.db` +
   `product_nppm_category_mapping.db` + this database.
4. Spot-check a handful of real `labels_tags` claim candidates (e.g.
   `en:no-added-sugar`, `en:low-or-no-salt`) against the matching `claim_rules` rows
   by hand, before trusting the automated match end-to-end.
